In [3]:
import sys
sys.path.append("../src")

from pathlib import Path
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor

from energy_load_forecast.ingestion import load_and_resample
from energy_load_forecast.preprocessing import clean_tail, add_calendar_features, chronological_split
from energy_load_forecast.features import add_all_features
from energy_load_forecast.evaluation import evaluate

In [4]:
series = load_and_resample(Path("../data/raw/entsoe/load_RO_2026-09-13_2026-09-20.csv"))
series_clean = clean_tail(series)
df = add_calendar_features(series_clean.to_frame())
df = add_all_features(df, lags=[1, 24, 168], windows=[3, 24])
df_copy = df.copy()

print("Shape:", df_copy.shape)
print(df_copy.head())

Shape: (143, 10)
                           load_mw  hour_local  day_of_week_local  \
timestamp_utc                                                       
2026-09-13 00:00:00+00:00  4417.25           3                  6   
2026-09-13 01:00:00+00:00  4406.50           4                  6   
2026-09-13 02:00:00+00:00  4544.25           5                  6   
2026-09-13 03:00:00+00:00  4611.25           6                  6   
2026-09-13 04:00:00+00:00  4719.75           7                  6   

                           is_weekend_local  month_local  load_mw_lag1  \
timestamp_utc                                                            
2026-09-13 00:00:00+00:00              True            9           NaN   
2026-09-13 01:00:00+00:00              True            9       4417.25   
2026-09-13 02:00:00+00:00              True            9       4406.50   
2026-09-13 03:00:00+00:00              True            9       4544.25   
2026-09-13 04:00:00+00:00              True            

In [5]:
train_df, val_df, test_df = chronological_split(
    df_copy,
    train_end="2026-09-16",
    val_end="2026-09-18"
)

X_train, y_train = train_df.drop(columns=["load_mw"]), train_df["load_mw"]
X_val, y_val = val_df.drop(columns=["load_mw"]), val_df["load_mw"]
X_test, y_test = test_df.drop(columns=["load_mw"]), test_df["load_mw"]

print("Train shape:", X_train.shape)
print("Val shape:  ", X_val.shape)
print("Test shape: ", X_test.shape)

Train shape: (72, 9)
Val shape:   (48, 9)
Test shape:  (23, 9)


In [6]:
model = LGBMRegressor(random_state=42)
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000024 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 117
[LightGBM] [Info] Number of data points in the train set: 72, number of used features: 7
[LightGBM] [Info] Start training from score 5141.819444
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [7]:
y_val_pred = np.asarray(model.predict(X_val)).reshape(-1)
print("Val predictions (first 5):", y_val_pred[:5])

Val predictions (first 5): [5025.33339768 5122.27802815 5122.27802815 5589.83838231 5711.96230182]


In [8]:
y_test_pred_arr = np.asarray(model.predict(X_test)).reshape(-1)
y_test_pred = pd.Series(y_test_pred_arr, index=y_test.index, name="forecast")
print("Test predictions:\n", y_test_pred)

Test predictions:
 timestamp_utc
2026-09-18 00:00:00+00:00    5325.947752
2026-09-18 01:00:00+00:00    5325.947752
2026-09-18 02:00:00+00:00    5325.947752
2026-09-18 03:00:00+00:00    5682.933074
2026-09-18 04:00:00+00:00    6015.635694
2026-09-18 05:00:00+00:00    5934.860786
2026-09-18 06:00:00+00:00    5832.675830
2026-09-18 07:00:00+00:00    5251.980258
2026-09-18 08:00:00+00:00    4898.223192
2026-09-18 09:00:00+00:00    4881.225593
2026-09-18 10:00:00+00:00    4962.148412
2026-09-18 11:00:00+00:00    4884.240350
2026-09-18 12:00:00+00:00    4711.211675
2026-09-18 13:00:00+00:00    4851.516483
2026-09-18 14:00:00+00:00    5840.444132
2026-09-18 15:00:00+00:00    6152.580341
2026-09-18 16:00:00+00:00    6166.040491
2026-09-18 17:00:00+00:00    6166.040491
2026-09-18 18:00:00+00:00    6166.040491
2026-09-18 19:00:00+00:00    6166.040491
2026-09-18 20:00:00+00:00    6123.935565
2026-09-18 21:00:00+00:00    5436.364895
2026-09-18 22:00:00+00:00    5072.098554
Freq: h, Name: forecast,

In [9]:
lgbm_scores = evaluate(y_test, y_test_pred)
print("LightGBM scores:", lgbm_scores)

LightGBM scores: {'MAE': 212.88063493291912, 'RMSE': 286.52100457967515, 'MAPE': 3.8544013286418695}


In [10]:
history = pd.concat([train_df["load_mw"], val_df["load_mw"]])
horizon = len(y_test)

last_value = history.iloc[-1]
naive_pred = pd.Series(last_value, index=y_test.index, name="forecast")

season_length = 24
seasonal_block = history.iloc[-season_length:]
n_tiles = -(-horizon // season_length)
seasonal_values = list(seasonal_block.to_numpy()) * n_tiles
seasonal_pred = pd.Series(seasonal_values[:horizon], index=y_test.index, name="forecast")

naive_scores = evaluate(y_test, naive_pred)
seasonal_scores = evaluate(y_test, seasonal_pred)

comparison = pd.DataFrame(
    [naive_scores, seasonal_scores, lgbm_scores],
    index=["Naive", "Seasonal Naive", "LightGBM"],
)
print("Comparison:\n", comparison)

Comparison:
                        MAE        RMSE       MAPE
Naive           645.739130  865.271671  10.854193
Seasonal Naive  235.010870  313.850036   4.581116
LightGBM        212.880635  286.521005   3.854401


In [11]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
print("Feature importances:\n", importances.sort_values(ascending=False))

Feature importances:
 load_mw_lag24               57
load_mw_rolling_mean_24h    35
hour_local                  33
load_mw_lag1                18
load_mw_rolling_mean_3h     14
month_local                  0
is_weekend_local             0
day_of_week_local            0
load_mw_lag168               0
dtype: int32
